In [1]:
import os, subprocess, glob, json, shutil
from datasets import load_dataset
from pathlib import Path

In [2]:
def find_patch_diff_glob(instance_id, base_dir, run_name):
    prefix_dir = os.path.join(
        base_dir,
        run_name,
        "final_eval",
        "logs",
        "run_evaluation",
    )

    pattern = os.path.join(prefix_dir, "**", instance_id, "patch.diff")
    matches = glob.glob(pattern, recursive=True)
    if not matches:
        raise FileNotFoundError(f"Could not find {instance_id} generated patch file")
    # newest by mtime
    matches.sort(key=lambda p: os.stat(p).st_mtime, reverse=True)
    found_file = matches[0]

    if os.path.exists(found_file) and os.access(found_file, os.R_OK):
        return matches[0]
    else:
        raise FileNotFoundError(f"Could not find {instance_id} generated patch file")

In [60]:
base_dir = "/workspaces/OpenHands/evaluation/evaluation_outputs/outputs/princeton-nlp__SWE-bench_Verified/CodeActAgent/"
run_name = "cepo_v8_1119_qwen480b_together_maxiter_500_N_10_verified"
swebench_hf_dataset = load_dataset("princeton-nlp/SWE-bench_Verified", split="test")

In [ ]:
instance_id = "psf__requests-2931"
hf_search_result = swebench_hf_dataset.filter(lambda x: x["instance_id"] == instance_id)[0]


problem_statement = hf_search_result["problem_statement"]
golden_patch = hf_search_result["patch"]
generated_patch_file = find_patch_diff_glob(instance_id, base_dir, run_name)
with open(generated_patch_file, "r", encoding="utf-8") as f:
    pred_patch = f.read()
test_patch = hf_search_result["test_patch"]
test_output_file = os.path.join(os.path.dirname(generated_patch_file), "test_output.txt")


marker = "============================= test session starts =============================="


with open(test_output_file) as f:
    for line in f:
        if line.strip() == marker:
            break
    test_outputs = f.read()   # read everything after marker
    test_outputs = marker + "\n" + test_outputs


chatgpt_prompt = f"""
I am running Swebench with openhands.

Here is the problem: {problem_statement}

========================================
Below is the ground truth golden patch: {golden_patch}

========================================
Below is my agent's generated patch: {pred_patch}

========================================
swebench applied the following test patch during evaluation
(Note this test patch is not accessible by the model at inference time)
(so when you make suggestions, you should not suggest the model to look at the hidden test, because this is not feasible)
Below is the test patch: {test_patch}

========================================
Test results and failures are here: {test_outputs}

========================================
First use a very simple sentence to describe this github issue in layman terms.
Then summarize in 2-3 sentences, very concisely but accurately, why my agent/s patch fails and why the golden patch succeed.
Then give some general suggestions on how I should prompt the agent to avoid such error.

"""

# I also attach the LLM trajectory, hope it is helpful, please look through it and tell me which key steps are failing.


In [62]:
print(chatgpt_prompt)


I am running Swebench with openhands.

Here is the problem: Request with binary payload fails due to calling to_native_string
Introduced with https://github.com/kennethreitz/requests/issues/2844

```
import requests
requests.put("http://httpbin.org/put", data=u"ööö".encode("utf-8"))
```

This works with 2.8.1, but not with 2.9.



Below is the ground truth golden patch: diff --git a/requests/models.py b/requests/models.py
--- a/requests/models.py
+++ b/requests/models.py
@@ -81,7 +81,7 @@ def _encode_params(data):
         """
 
         if isinstance(data, (str, bytes)):
-            return to_native_string(data)
+            return data
         elif hasattr(data, 'read'):
             return data
         elif hasattr(data, '__iter__'):
@@ -385,6 +385,9 @@ def prepare_url(self, url, params):
             if isinstance(fragment, str):
                 fragment = fragment.encode('utf-8')
 
+        if isinstance(params, (str, bytes)):
+            params = to_native_string(params)
+


In [63]:
print(golden_patch)

diff --git a/requests/models.py b/requests/models.py
--- a/requests/models.py
+++ b/requests/models.py
@@ -81,7 +81,7 @@ def _encode_params(data):
         """
 
         if isinstance(data, (str, bytes)):
-            return to_native_string(data)
+            return data
         elif hasattr(data, 'read'):
             return data
         elif hasattr(data, '__iter__'):
@@ -385,6 +385,9 @@ def prepare_url(self, url, params):
             if isinstance(fragment, str):
                 fragment = fragment.encode('utf-8')
 
+        if isinstance(params, (str, bytes)):
+            params = to_native_string(params)
+
         enc_params = self._encode_params(params)
         if enc_params:
             if query:



In [64]:
print(pred_patch)

diff --git a/requests/models.py b/requests/models.py
index 9c624d3..8bce13c 100644
--- a/requests/models.py
+++ b/requests/models.py
@@ -80,8 +80,10 @@ class RequestEncodingMixin(object):
         if parameters are supplied as a dict.
         """
 
-        if isinstance(data, (str, bytes)):
+        if isinstance(data, str):
             return to_native_string(data)
+        elif isinstance(data, bytes):
+            return data
         elif hasattr(data, 'read'):
             return data
         elif hasattr(data, '__iter__'):



In [58]:
print(problem_statement)

Request with binary payload fails due to calling to_native_string
Introduced with https://github.com/kennethreitz/requests/issues/2844

```
import requests
requests.put("http://httpbin.org/put", data=u"ööö".encode("utf-8"))
```

This works with 2.8.1, but not with 2.9.




In [59]:
print(hf_search_result["test_patch"])

diff --git a/test_requests.py b/test_requests.py
--- a/test_requests.py
+++ b/test_requests.py
@@ -157,6 +157,11 @@ def test_params_bytes_are_encoded(self):
                                    params=b'test=foo').prepare()
         assert request.url == 'http://example.com/?test=foo'
 
+    def test_binary_put(self):
+        request = requests.Request('PUT', 'http://example.com',
+                                   data=u"ööö".encode("utf-8")).prepare()
+        assert isinstance(request.body, bytes)
+
     def test_mixed_case_scheme_acceptable(self, httpbin):
         s = requests.Session()
         s.proxies = getproxies()

